# S8.4 — Methodology & Findings Deep-Dive

Verifies the `MethodologyService` that extracts structured methodology analysis from papers:
- Research design, datasets, baselines, key results, statistical significance, reproducibility
- Content preparation with methodology-focused section prioritization
- API endpoint `POST /api/v1/papers/{paper_id}/methodology`
- Error handling and fallback parsing

In [1]:
import sys, os, nest_asyncio, asyncio
sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), "../..")))
nest_asyncio.apply()
print("✓ Path configured, nest_asyncio applied")

✓ Path configured, nest_asyncio applied


## 1. Schema Validation

In [2]:
from src.schemas.api.analysis import (
    DatasetInfo, ResultEntry, MethodologyAnalysis, MethodologyResponse,
)
import uuid

# DatasetInfo
ds = DatasetInfo(name="WMT 2014 EN-DE", description="Translation benchmark", size="4.5M pairs")
assert ds.name == "WMT 2014 EN-DE"
assert ds.size == "4.5M pairs"
print(f"✓ DatasetInfo: {ds.model_dump()}")

# ResultEntry
re = ResultEntry(metric="BLEU", value="28.4", context="WMT 2014 EN-DE")
assert re.metric == "BLEU"
print(f"✓ ResultEntry: {re.model_dump()}")

# MethodologyAnalysis — full
analysis = MethodologyAnalysis(
    research_design="Experimental study on machine translation.",
    datasets=[ds],
    baselines=["GNMT", "ConvS2S"],
    key_results=[re],
    statistical_significance="Exceeds SOTA by 2+ BLEU",
    reproducibility_notes="8 P100 GPUs, 3.5 days",
)
assert len(analysis.datasets) == 1
assert len(analysis.key_results) == 1
print(f"✓ MethodologyAnalysis (full): {len(analysis.datasets)} datasets, {len(analysis.key_results)} results")

# MethodologyAnalysis — empty (theoretical paper)
empty = MethodologyAnalysis(research_design="Theoretical analysis.")
assert empty.datasets == []
assert empty.key_results == []
assert empty.statistical_significance is None
print("✓ MethodologyAnalysis (theoretical/empty): validated")

# MethodologyResponse
pid = uuid.UUID("12345678-1234-5678-1234-567812345678")
resp = MethodologyResponse(
    paper_id=pid, analysis=analysis,
    model="gemini-3-flash", provider="gemini", latency_ms=1100.0,
)
assert resp.paper_id == pid
assert resp.model == "gemini-3-flash"
print(f"✓ MethodologyResponse: paper_id={resp.paper_id}, model={resp.model}")

print("\n✓ All schema validations passed")

✓ DatasetInfo: {'name': 'WMT 2014 EN-DE', 'description': 'Translation benchmark', 'size': '4.5M pairs'}
✓ ResultEntry: {'metric': 'BLEU', 'value': '28.4', 'context': 'WMT 2014 EN-DE'}
✓ MethodologyAnalysis (full): 1 datasets, 1 results
✓ MethodologyAnalysis (theoretical/empty): validated
✓ MethodologyResponse: paper_id=12345678-1234-5678-1234-567812345678, model=gemini-3-flash

✓ All schema validations passed


## 2. Service — Full Paper Analysis (Mocked LLM)

In [3]:
import json
from unittest.mock import AsyncMock, MagicMock
from src.services.analysis.methodology import MethodologyService
from src.services.llm.provider import LLMResponse, UsageMetadata

PAPER_ID = uuid.UUID("12345678-1234-5678-1234-567812345678")

mock_json = json.dumps({
    "research_design": "Experimental study on machine translation using attention.",
    "datasets": [
        {"name": "WMT 2014 EN-DE", "description": "Translation benchmark", "size": "4.5M pairs"},
    ],
    "baselines": ["GNMT", "ConvS2S", "ByteNet"],
    "key_results": [
        {"metric": "BLEU", "value": "28.4", "context": "WMT 2014 EN-DE"},
        {"metric": "BLEU", "value": "41.0", "context": "WMT 2014 EN-FR"},
    ],
    "statistical_significance": "Exceeds SOTA by >2 BLEU points.",
    "reproducibility_notes": "8 P100 GPUs, 3.5 days training.",
})

# Mock paper
paper = MagicMock()
paper.id = PAPER_ID
paper.title = "Attention Is All You Need"
paper.authors = ["Vaswani et al."]
paper.abstract = "We propose the Transformer, based entirely on attention mechanisms."
paper.sections = [
    {"title": "Methodology", "content": "Multi-head attention with scaled dot-product.", "level": 1},
    {"title": "Experiments", "content": "Trained on WMT 2014 EN-DE and EN-FR.", "level": 1},
    {"title": "Results", "content": "28.4 BLEU on EN-DE, 41.0 on EN-FR.", "level": 1},
]
paper.pdf_content = "Full text content."

# Mock deps
mock_repo = AsyncMock()
mock_repo.get_by_id = AsyncMock(return_value=paper)

mock_llm = AsyncMock()
mock_llm.generate = AsyncMock(return_value=LLMResponse(
    text=mock_json, model="gemini-3-flash", provider="gemini",
    usage=UsageMetadata(prompt_tokens=600, completion_tokens=300, total_tokens=900, latency_ms=1100.0),
))

service = MethodologyService(llm_provider=mock_llm, paper_repo=mock_repo)
result = asyncio.get_event_loop().run_until_complete(service.analyze_methodology(PAPER_ID))

a = result.analysis
assert a.research_design
assert len(a.datasets) >= 1
assert len(a.baselines) >= 1
assert len(a.key_results) >= 1
assert a.statistical_significance is not None
assert a.reproducibility_notes is not None
assert result.warning is None

print(f"Research design: {a.research_design}")
print(f"Datasets: {[d.name for d in a.datasets]}")
print(f"Baselines: {a.baselines}")
print(f"Key results: {[(r.metric, r.value) for r in a.key_results]}")
print(f"Statistical significance: {a.statistical_significance}")
print(f"Reproducibility: {a.reproducibility_notes}")
print(f"Model: {result.model}, Latency: {result.latency_ms}ms")
print("\n✓ Full paper methodology analysis passed")

Research design: Experimental study on machine translation using attention.
Datasets: ['WMT 2014 EN-DE']
Baselines: ['GNMT', 'ConvS2S', 'ByteNet']
Key results: [('BLEU', '28.4'), ('BLEU', '41.0')]
Statistical significance: Exceeds SOTA by >2 BLEU points.
Reproducibility: 8 P100 GPUs, 3.5 days training.
Model: gemini-3-flash, Latency: 1100.0ms

✓ Full paper methodology analysis passed


## 3. Abstract-Only & Theoretical Paper Edge Cases

In [4]:
# Abstract-only paper -> warning
paper_abs = MagicMock()
paper_abs.id = PAPER_ID
paper_abs.title = "Attention Is All You Need"
paper_abs.authors = ["Vaswani et al."]
paper_abs.abstract = "We propose the Transformer, based entirely on attention mechanisms."
paper_abs.sections = []
paper_abs.pdf_content = None

repo_abs = AsyncMock()
repo_abs.get_by_id = AsyncMock(return_value=paper_abs)

svc_abs = MethodologyService(llm_provider=mock_llm, paper_repo=repo_abs)
res_abs = asyncio.get_event_loop().run_until_complete(svc_abs.analyze_methodology(PAPER_ID))
assert res_abs.warning is not None
assert "abstract" in res_abs.warning.lower()
print(f"✓ Abstract-only warning: {res_abs.warning}")

# Theoretical paper -> empty datasets/results
theoretical_json = json.dumps({
    "research_design": "Theoretical analysis of self-attention expressiveness.",
    "datasets": [],
    "baselines": [],
    "key_results": [],
    "statistical_significance": None,
    "reproducibility_notes": "Proofs in appendix.",
})
mock_llm_theo = AsyncMock()
mock_llm_theo.generate = AsyncMock(return_value=LLMResponse(
    text=theoretical_json, model="gemini-3-flash", provider="gemini",
    usage=UsageMetadata(latency_ms=900.0),
))

svc_theo = MethodologyService(llm_provider=mock_llm_theo, paper_repo=mock_repo)
res_theo = asyncio.get_event_loop().run_until_complete(svc_theo.analyze_methodology(PAPER_ID))
assert "theoretical" in res_theo.analysis.research_design.lower()
assert len(res_theo.analysis.datasets) == 0
assert len(res_theo.analysis.key_results) == 0
print(f"✓ Theoretical paper: design='{res_theo.analysis.research_design[:60]}...', 0 datasets, 0 results")

print("\n✓ Edge cases passed")

✓ Abstract-only warning: Only abstract available — methodology analysis extracted from abstract alone
✓ Theoretical paper: design='Theoretical analysis of self-attention expressiveness....', 0 datasets, 0 results

✓ Edge cases passed


## 4. Error Handling

In [5]:
from src.exceptions import PaperNotFoundError, InsufficientContentError, LLMServiceError

# Paper not found
repo_none = AsyncMock()
repo_none.get_by_id = AsyncMock(return_value=None)
svc_nf = MethodologyService(llm_provider=mock_llm, paper_repo=repo_none)
try:
    asyncio.get_event_loop().run_until_complete(svc_nf.analyze_methodology(PAPER_ID))
    assert False, "Should have raised PaperNotFoundError"
except PaperNotFoundError:
    print("✓ PaperNotFoundError raised for missing paper")

# Insufficient content
paper_empty = MagicMock()
paper_empty.id = PAPER_ID
paper_empty.abstract = ""
paper_empty.sections = []
paper_empty.pdf_content = None
repo_empty = AsyncMock()
repo_empty.get_by_id = AsyncMock(return_value=paper_empty)
svc_ic = MethodologyService(llm_provider=mock_llm, paper_repo=repo_empty)
try:
    asyncio.get_event_loop().run_until_complete(svc_ic.analyze_methodology(PAPER_ID))
    assert False, "Should have raised InsufficientContentError"
except InsufficientContentError:
    print("✓ InsufficientContentError raised for empty paper")

# LLM failure
mock_llm_fail = AsyncMock()
mock_llm_fail.generate = AsyncMock(side_effect=Exception("API down"))
svc_fail = MethodologyService(llm_provider=mock_llm_fail, paper_repo=mock_repo)
try:
    asyncio.get_event_loop().run_until_complete(svc_fail.analyze_methodology(PAPER_ID))
    assert False, "Should have raised LLMServiceError"
except LLMServiceError:
    print("✓ LLMServiceError raised on LLM failure")

# Malformed LLM output -> fallback
mock_llm_bad = AsyncMock()
mock_llm_bad.generate = AsyncMock(return_value=LLMResponse(
    text="Not valid JSON at all", model="gemini-3-flash", provider="gemini",
    usage=UsageMetadata(latency_ms=500.0),
))
svc_bad = MethodologyService(llm_provider=mock_llm_bad, paper_repo=mock_repo)
res_bad = asyncio.get_event_loop().run_until_complete(svc_bad.analyze_methodology(PAPER_ID))
assert res_bad.warning is not None
assert "malformed" in res_bad.warning.lower()
print(f"✓ Malformed output fallback: {res_bad.warning}")

print("\n✓ All error handling tests passed")

Failed to parse methodology analysis JSON, using fallback: Not valid JSON at all


✓ PaperNotFoundError raised for missing paper
✓ InsufficientContentError raised for empty paper
✓ LLMServiceError raised on LLM failure
✓ Malformed output fallback: LLM returned malformed output — fallback analysis generated

✓ All error handling tests passed


## 5. Content Preparation & Truncation

In [6]:
# Content preparation prioritizes methodology sections
svc = MethodologyService(llm_provider=mock_llm, paper_repo=mock_repo)
content = svc._prepare_content(paper)
assert "Methodology" in content
assert "Results" in content
assert "Experiments" in content
print(f"✓ Content preparation includes priority sections (len={len(content)} chars)")

# Truncation to ~4000 words
long_paper = MagicMock()
long_paper.abstract = "Abstract text."
long_paper.sections = [{"title": "Methods", "content": " ".join(["word"] * 10000), "level": 1}]
long_paper.pdf_content = None

content_long = svc._prepare_content(long_paper)
word_count = len(content_long.split())
assert word_count <= 4500
print(f"✓ Truncation: {word_count} words (limit ~4000)")

print("\n✓ Content preparation tests passed")

✓ Content preparation includes priority sections (len=241 chars)
✓ Truncation: 4000 words (limit ~4000)

✓ Content preparation tests passed


## 6. API Endpoint Test

In [7]:
from httpx import ASGITransport, AsyncClient
from src.dependency import get_llm_provider, get_paper_repository
from src.main import create_app

app = create_app()
app.dependency_overrides[get_paper_repository] = lambda: mock_repo
app.dependency_overrides[get_llm_provider] = lambda: mock_llm

async def _test_endpoint():
    async with AsyncClient(transport=ASGITransport(app=app), base_url="http://test") as client:
        # Success
        resp = await client.post(f"/api/v1/papers/{PAPER_ID}/methodology")
        assert resp.status_code == 200
        data = resp.json()
        assert "analysis" in data
        assert "research_design" in data["analysis"]
        assert "datasets" in data["analysis"]
        assert "key_results" in data["analysis"]
        assert "baselines" in data["analysis"]
        print(f"✓ POST 200: research_design='{data['analysis']['research_design'][:50]}...'")

        # 404
        app.dependency_overrides[get_paper_repository] = lambda: repo_none
        resp404 = await client.post(f"/api/v1/papers/{PAPER_ID}/methodology")
        assert resp404.status_code == 404
        print(f"✓ POST 404: {resp404.json()['detail'][:50]}")

    app.dependency_overrides.clear()

asyncio.get_event_loop().run_until_complete(_test_endpoint())

print("\n✓ API endpoint tests passed")

✓ POST 200: research_design='Experimental study on machine translation using at...'
✓ POST 404: Paper 12345678-1234-5678-1234-567812345678 not fou

✓ API endpoint tests passed
